# Train CSRNet on Colab

Owner: Ronish. Runtime → Change runtime type → **T4 GPU** before running.

Before the first run, upload the ShanghaiTech zip from Kaggle (`archive.zip`, ~349 MB) to
`MyDrive/crowdstrike/archive.zip`. Checkpoints and logs are saved to `MyDrive/crowdstrike/checkpoints`,
so they survive a disconnect.

## 1. Settings

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/crowdstrike"
ZIP_PATH = f"{DRIVE_DIR}/archive.zip"
CKPT_DIR = f"{DRIVE_DIR}/checkpoints"
BRANCH = "main"        # use "ronish/model-training" until it is merged
PART = "B"
EPOCHS = 100           # EarlyStopping usually stops sooner; keep total GPU time under ~4 h
BATCH_SIZE = 8
RESUME = ""            # e.g. f"{CKPT_DIR}/csrnet_partB_best.weights.h5" after a disconnect

## 2. GPU check and Google Drive

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

from google.colab import drive
drive.mount("/content/drive")

## 3. Clone the repo

In [ ]:
%cd /content
!rm -rf CrowdStrike
!git clone -q -b {BRANCH} https://github.com/Prajwal382/CrowdStrike.git
%cd /content/CrowdStrike
!git log --oneline -3
# Colab already has TensorFlow, NumPy, SciPy, OpenCV, Matplotlib and pandas installed.

## 4. Dataset and density maps
Unzip from Drive into local disk (faster than reading from Drive), then rebuild the maps.

In [ ]:
!mkdir -p data
!unzip -q -o "{ZIP_PATH}" "ShanghaiTech/*" -d data
!python scripts/check_dataset.py --part {PART}
!python scripts/make_density_maps.py --part {PART}

## 5. Quick smoke test (about 1 minute)

In [ ]:
!python src/train.py --smoke --out-dir /content/smoke_ckpt --figure-dir /content/smoke_fig

## 6. Train

In [ ]:
resume_flag = f"--resume {RESUME}" if RESUME else ""
!python src/train.py --part {PART} --epochs {EPOCHS} --batch-size {BATCH_SIZE} \
    --out-dir {CKPT_DIR} --figure-dir {CKPT_DIR} {resume_flag}

## 7. Training curve

In [ ]:
from IPython.display import Image, display
display(Image(f"{CKPT_DIR}/training_curve_part{PART}.png"))

## 8. Hand-off

- Best weights: `MyDrive/crowdstrike/checkpoints/csrnet_partB_best.weights.h5` → share with the team for `src/evaluate.py --weights ...`.
- Copy `training_curve_partB.png` into `outputs/figures/` in the repo for the report.